# Topic 09 — Apply / Map / Transform

> **Investigation milestone:** Time to engineer the analyst features the capstone
> needs: customer segments, margin bands, delivery-late flags, RFM-style fields.
> And to learn *when not to use `apply`* — because on Aurora's 64k lines, the
> wrong choice is 100× slower.

**Time split: 20% reading · 80% in `practice.ipynb`. NumPy recall checkpoint at the end.**

---

## The toolbox, fastest → slowest

1. **Vectorized ops / `np.where` / `np.select`** — always try first.
2. **`map`** (Series) — element-wise via a dict or function. Great for lookups.
3. **`apply`** (Series/DataFrame) — arbitrary Python per element/row. Flexible, slow.
4. **`transform`** (groupby) — group stat broadcast back to rows (Topic 5).

```python
# map: dictionary lookup (fast, clear)
seg = {"consumer": "B2C", "business": "B2B", "pro athlete": "B2C"}
customers["seg2"] = customers["segment"].str.lower().map(seg)

# np.where: vectorized if/else
items["margin_band"] = np.where(items["line_profit"] > 0, "profit", "loss")

# np.select: multi-branch
cond = [items["quantity"] >= 100, items["quantity"] >= 10]
items["size_band"] = np.select(cond, ["bulk", "medium"], default="small")
```

## When `apply` is a code smell

```python
# ❌ slow: row-wise Python
df["rev"] = df.apply(lambda r: r["q"] * r["p"], axis=1)
# ✅ fast: vectorized
df["rev"] = df["q"] * df["p"]
```
`apply(axis=1)` runs a Python function per row — it defeats vectorization. Reach
for it only when the logic genuinely can't be expressed with column math,
`np.where`, `np.select`, or `map`. **Never** use `iterrows` for math.

## `map` vs `apply` vs `applymap`
- `Series.map` — element-wise, takes a dict **or** function. Best for lookups.
- `Series.apply` — element-wise function (no dict).
- `DataFrame.apply` — per column (`axis=0`) or per row (`axis=1`).
- `DataFrame.map` (was `applymap`) — element-wise over every cell.

## NumPy connection 🔢

This whole topic *is* the NumPy lesson: vectorization vs Python loops,
`np.where`, `np.select`, broadcasting. The performance gap (often 50–200×) comes
from staying in compiled NumPy land instead of bouncing into the Python
interpreter once per row. Time it yourself in practice.

## Visual learning 📊

After feature engineering, a stacked bar of orders by `size_band` × `channel`,
or a histogram of `margin_band`, validates that your new features behave sanely.

---

## 🔎 Interview Lens (answer in writing)
- **Q26:** Why is vectorization faster than `iterrows`/loops? What's happening underneath?
- **Q27:** When is `apply` a code smell, and what replaces it?

### Recap
1. Order the toolbox fastest → slowest.
2. Rewrite a row-wise `apply` multiply as vectorized.
3. `map` vs `apply` on a Series — the key difference?

➡️ Open **`practice.ipynb`** (NumPy recall checkpoint 3).